In [1]:
# [START analyze_sentiment]
import os
import csv
from azure.core.credentials import AzureKeyCredential
from azure.ai.textanalytics import TextAnalyticsClient
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler

In [2]:
# endpoint = os.environ["LANGUAGE_ENDPOINT"]
# key = os.environ["LANGUAGE_KEY"]
#
# client = TextAnalyticsClient(endpoint=endpoint, credential=AzureKeyCredential(key))

In [3]:
# sentence="By choosing a bike over a car, I’m reducing my environmental footprint. Cycling promotes eco-friendly transportation, and I’m proud to be part of that movement.."
#
# result=client.analyze_sentiment([sentence],show_opinion_mining=True)
# print(result)

In [4]:
# load some data
crtDir =  os.getcwd()
fileName = os.path.join(crtDir, 'data', 'reviews_mixed.csv')

data = []
with open(fileName) as csv_file:
    csv_reader = csv.reader(csv_file, delimiter=',')
    line_count = 0
    for row in csv_reader:
        if line_count == 0:
            dataNames = row
        else:
            data.append(row)
        line_count += 1

inputs = [data[i][0] for i in range(len(data))][:150]
outputs = [data[i][1] for i in range(len(data))][:150]
labelNames = list(set(outputs))

In [5]:
# prepare data for training and testing

import numpy as np

np.random.seed(5)
# noSamples = inputs.shape[0]
noSamples = len(inputs)
indexes = [i for i in range(noSamples)]
trainSample = np.random.choice(indexes, int(0.8 * noSamples), replace = False)
testSample = [i for i in indexes  if not i in trainSample]

trainInputs = [inputs[i] for i in trainSample]
trainOutputs = [outputs[i] for i in trainSample]
testInputs = [inputs[i] for i in testSample]
testOutputs = [outputs[i] for i in testSample]

In [6]:
# extragere caracteristici din texte
# 1. Bag of Words

from sklearn.feature_extraction.text import CountVectorizer
vectorizer = CountVectorizer()

trainFeatures = vectorizer.fit_transform(trainInputs)
testFeatures = vectorizer.transform(testInputs)

# vocab size
print("vocabulary size: ", len(vectorizer.vocabulary_),"words")
# samples
print("training data size: ",len(trainInputs),"samples")
# shape of feature matrix
print("trainFeatures shape: ",trainFeatures.shape)

# train data vocab
print("some words from train vocab: ",vectorizer.get_feature_names_out()[-20:])
# extracted features
print("some features: ", trainFeatures.toarray()[:3])

vocabulary size:  497 words
training data size:  120 samples
trainFeatures shape:  (120, 497)
some words from train vocab:  ['were' 'werent' 'wet' 'when' 'which' 'whole' 'whomever' 'wifi' 'window'
 'windows' 'winter' 'with' 'work' 'working' 'workout' 'would' 'years'
 'yet' 'you' 'your']
some features:  [[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]


In [7]:
# 2. TF-IDF (word granularity)
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer(max_features=50)

trainFeatures = vectorizer.fit_transform(trainInputs)
testFeatures= vectorizer.transform(testInputs)

# vocab from train data
print("vocab: ", vectorizer.get_feature_names_out()[:10])
# extracted features
print("features: ", trainFeatures.toarray()[:3])

vocab:  ['air' 'all' 'and' 'are' 'bathroom' 'bed' 'clean' 'coffee' 'comfortable'
 'facilities']
features:  [[0.         0.         0.         0.         0.         0.
  0.4794876  0.         0.         0.57987031 0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.43782502 0.
  0.         0.         0.49208873 0.         0.         0.
  0.         0.        ]
 [0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.41139464 0.         0.         0.         0.32401671
  0.         0.         0.         0.         0.         0.36364303
  0.         0.42380294 0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.        

In [8]:
# 3.  word2vec w pretrained model

import gensim.downloader as api

word2vecModel300=api.load('word2vec-google-news-300')

def get_mean_vector(text):
    words=str(text).lower().split()
    vectors = [word2vecModel300[word] for word in words if word in word2vecModel300]

    if not vectors:
        return np.zeros(300)

    return np.mean(vectors,axis=0)

trainFeatures=np.array([get_mean_vector(text) for text in trainInputs])
testFeatures=np.array([get_mean_vector(text) for text in testInputs])

# check results
print("word2vec features shape: ", trainFeatures.shape)
print("first review features (5): ", trainFeatures[0][:5])

word2vec features shape:  (120, 300)
first review features (5):  [-0.11783854  0.08870443  0.16292317 -0.00341797 -0.08015951]


In [9]:
# 4. FastText w pretrained model ( other method )
# Word2Vec fails on typos ( like unconfortable ) or slang ( like soooo ) because it looks for the whole word.
# FastText succeeds because it looks at "n-grams" (chunks of characters).

from gensim.models import FastText

tokenized_reviews=[str(text).lower().split() for text in trainInputs]

model_ft=FastText(sentences=tokenized_reviews, vector_size=100,window=5, min_count=1)

def get_sentence_features(text):
    # standardize text
    words=str(text).lower().split()

    # extract vectors for words that the google model knows
    vectors=[model_ft.wv[w] for w in words if w in model_ft.wv]

    # if review is empty or has no known words return zeros
    if len(vectors)==0:
        return np.zeros(model_ft.vector_size)

    # avg the vectors along the first axis
    return np.mean(vectors,axis=0)

trainFeatures=[get_sentence_features(text) for text in trainInputs]
testFeatures=[get_sentence_features(text) for text in testInputs]

# convert to numpy matrix
X_train=np.array(trainFeatures)
X_test=np.array(testFeatures)

# (number_of_reviews, vector_size)
print(f"Train matrix shape: {X_train.shape}")
# features of first 2 reviews, first 5 dimensions
print("Extracted Features (Preview):")
print(X_train[:2, :5])

print("\nTesting 'OutOfVocabulary' extraction for 'soooo':")
if 'soooo' in model_ft.wv:
    print("Vector for 'soooo' exists!")
else:
    # FastText creates vectors even if the word isn't in vocab!
    print("Vector generated via sub-words:", model_ft.wv['soooo'][:5])

Train matrix shape: (120, 100)
Extracted Features (Preview):
[[ 1.5537849e-03 -8.0595305e-04  1.0435412e-03  1.6415967e-03
   1.0575447e-03]
 [-2.4120638e-04  7.0987758e-04  4.7358877e-05  1.0545168e-04
   4.9446209e-04]]

Testing 'OutOfVocabulary' extraction for 'soooo':
Vector for 'soooo' exists!


In [10]:
# ANN cu tool

from sklearn.preprocessing import StandardScaler

scaler=StandardScaler()
X_train_scaled=scaler.fit_transform(X_train)
X_test_scaled=scaler.transform(X_test)

from sklearn.neural_network import MLPClassifier

# now try 10,5 and see how bad it is
clf=MLPClassifier(hidden_layer_sizes=(6,7),solver='lbfgs',max_iter=1000,random_state=67)

clf.fit(X_train_scaled,trainOutputs)

computedTestOutputs=clf.predict(X_test_scaled)
print(computedTestOutputs)

from sklearn.metrics import classification_report
print(classification_report(testOutputs, computedTestOutputs))

['negative' 'negative' 'negative' 'positive' 'positive' 'negative'
 'negative' 'negative' 'negative' 'negative' 'negative' 'negative'
 'positive' 'positive' 'negative' 'negative' 'negative' 'negative'
 'negative' 'positive' 'negative' 'negative' 'negative' 'negative'
 'negative' 'positive' 'positive' 'positive' 'negative' 'negative']
              precision    recall  f1-score   support

    negative       0.77      0.94      0.85        18
    positive       0.88      0.58      0.70        12

    accuracy                           0.80        30
   macro avg       0.82      0.76      0.77        30
weighted avg       0.81      0.80      0.79        30



In [11]:
# ann cu tool pe fraza: By choosing a bike over a car, I’m reducing my environmental footprint. Cycling promotes eco-friendly transportation, and I’m proud to be part of that movement..

sentence="By choosing a bike over a car, I’m reducing my environmental footprint. Cycling promotes eco-friendly transportation, and I’m proud to be part of that movement.."

sentenceFeatures=get_sentence_features(sentence)

sentenceFeatures=scaler.transform([sentenceFeatures])

predictedLabel=clf.predict(sentenceFeatures)
proba=clf.predict_proba(sentenceFeatures)

print("Predicted label: ", predictedLabel[0], proba[0,1])

Predicted label:  positive 0.9999896936439651


In [22]:
# ann manual

from ANNie import ANN

clf=ANN(input_size=100,hidden_size=5,output_size=1,epochs=1000,learning_rate=0.01)

# convert 'negative' to 0 and 'positive' to 1 ( so y-prediction can work instead of doing string - float  )
trainOutputsNumeric=np.array([1 if label == 'positive' else 0 for label in trainOutputs])

clf.fit(X_train_scaled,trainOutputsNumeric)

computedTestOutputs=clf.predict(X_test_scaled)

# same conversion for test outputs
testOutputsNumeric = [1 if label == 'positive' else 0 for label in testOutputs]

from sklearn.metrics import classification_report
print(classification_report(testOutputsNumeric, computedTestOutputs))

              precision    recall  f1-score   support

           0       0.81      0.72      0.76        18
           1       0.64      0.75      0.69        12

    accuracy                           0.73        30
   macro avg       0.73      0.74      0.73        30
weighted avg       0.74      0.73      0.74        30



In [23]:
# ann manual pe fraza: By choosing a bike over a car, I’m reducing my environmental footprint. Cycling promotes eco-friendly transportation, and I’m proud to be part of that movement..

sentence="By choosing a bike over a car, I’m reducing my environmental footprint. Cycling promotes eco-friendly transportation, and I’m proud to be part of that movement.."

sentenceFeatures=get_sentence_features(sentence)

sentenceFeatures=scaler.transform([sentenceFeatures])

predictedLabel=clf.predict(sentenceFeatures)
proba=clf.predict_proba(sentenceFeatures)

print("Predicted label: ", predictedLabel[0],proba)

Predicted label:  [1] [[0.90414814]]
